# Saving/running data with H5 files

In the following example one propose to save the channels during acquisition in a H5 (HDF5 format) file:

Import the packages:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.mu import Megamicros
from megamicros.core.h5 import MemsArrayH5

# set log level to INFO to get more information (available levels are DEBUG, INFO, WARNING, ERROR, FATAL)
log.setLevel( "WARNING" ) 

## Saving data in H5 file

In [ ]:
# Define an empty antenna
log.setLevel( "WARNING" )
antenna = Megamicros()

# Set all MEMs as available
antenna.setAvailableMems( [i for i in range(32)] )

antenna.run( 
    duration=4, 
    mems = antenna.available_mems,
    counter=False, 
    status=False,
    sampling_frequency=50000,
    h5_recording=True,                          # H5 recording ON
    h5_rootdir='./',                            # directory where to save file
    h5_filename='test.h5',                      # filename
)

antenna.wait()

### Control the H5 file created

We can control easily the H5 file generated by building a new antenna with this H5 file as input. 

In [ ]:
# Define the antenna
antenna_h5 = MemsArrayH5( 
    filename='test.h5',
)

print( f"Sampling frequency: {antenna_h5.sampling_frequency}Hz" )
print( f"Channels number: {antenna_h5.mems_number}" )
print( f"Available MEMs number: {antenna_h5.available_mems_number}" )
print( f"Available MEMs: {antenna_h5.available_mems}" )
print( f"Whether counter is available or not: {antenna_h5.counter}" )
print( f"Dataset(s) number: " + str(antenna_h5.dataset_number) )

### Plot signals

In [ ]:
print(antenna_h5.clear())

# Choose MEMS to plot
MEMS = [0, 1]

# Init a np.ndarray
signals = np.ndarray( (0, len(MEMS) ) )

# Run the H5 antenna
antenna_h5.run( 
    duration=1, 
    mems = MEMS,
    real_time=False,
)

# Get signals
for data in antenna_h5:
    # print(np.shape(data))
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna_h5.wait()

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna_h5.sampling_frequency
fig, axs = plt.subplots( antenna_h5.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna_h5.channels_number ):
    axs[s].plot( time, signals[:len(time),s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()